In [0]:
%run ../My-Folder/(Clone)-Copy-Datasets

In [0]:
%sql
select * from customers;

In [0]:
%sql
describe customers

In [0]:
%sql
select customer_id, profile: first_name, profile: address: country
from customers
where profile: address: country in ('Kosovo','Albania');

In [0]:
%sql
create or replace temp view parsed_customers as
select customer_id, from_json(profile, schema_of_json('{"first_name":"Ronna","last_name":"Gonning","gender":"Non-binary","address":{"street":"48 Grim Way","city":"Metsemotlhaba","country":"Botswana"}}')) as profile_struct
from customers;

select * from parsed_customers;


In [0]:
%sql
select customer_id, profile_struct.address.country
from parsed_customers

In [0]:
%sql
with flattened as (
    select customer_id, profile_struct.* 
    from parsed_customers
)
select customer_id, first_name, last_name, gender, address.* from flattened;


In [0]:
%sql
select *
from orders
limit 5;

In [0]:
%sql
with exploded as (
    select order_id, customer_id, order_timestamp, explode(books) as book
    from orders
)
select order_id, customer_id, book.book_id, book.quantity as quantity, book.subtotal as price, order_timestamp
from exploded;

In [0]:
%sql
select customer_id, collect_set(books.book_id) as unique_purchased_books
from orders
group by customer_id
limit 5;

In [0]:
%sql
select * from orders where customer_id = 'C00001'

In [0]:
%sql
-- The CTE first collects all the unique permutations of book IDs across orders per customer. The result is different arrays which match to the number of orders with *unique permutations*.So, if a customer has made five orders, and in three of them the same books were purchased, in the same sequence, then we'll have three unique permutations of books (book_id), and collect_set(books.book_id) results into a cell with three arrays holding those book_id permutations.

WITH collected AS (
    SELECT customer_id, collect_set(books.book_id) AS unique_purchased_books
    FROM orders
    GROUP BY customer_id
)
SELECT customer_id, flatten(unique_purchased_books)
FROM collected;

In [0]:
%sql
-- The CTE first collects the unique book-ID permutations across orders, grouped by customer. In cases where a customer has 2+ orders that contain a unique sequence of book_id, then we'll have cells which store an array of arrays that have one or more books (book_id's). As the query continues, the CTE is utilized, and those cells with multiple arrays, are flattened into a single array, and from there we pick up the unique books.
with collected as (
    select customer_id, collect_set(books.book_id) as unique_purchased_books
    from orders
    group by customer_id
)
select customer_id, array_distinct(flatten(unique_purchased_books))
from collected


In [0]:
%sql
with exploded_orders as (
    select *, explode(books) as book from orders
) 
select order_id, customer_id, book_id, title, author, price, order_timestamp from exploded_orders eo 
left join books.csv_books b
on b.book_id = eo.book.book_id